In [8]:
# 1. Load the dataset into a Pandas DataFrame. 
import pandas as pd

df = pd.read_csv('data/patient_clinical_data_raw.csv')

# 2. Display the first 10 and last 10 records.
print("First 10 rows of the dataset:")
print(df.head(10))
print("Last 10 rows of the dataset:")
print(df.tail(10))

# 3. Find the number of rows and columns.
rows, columns = df.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {columns}")

# 4. Display the column names and their data types.
print(df.dtypes)

# 5. Generate a statistical summary of the numerical columns.
print(df.describe())

# 6. Identify the number of unique values in each column. 
print(df.nunique())

First 10 rows of the dataset:
  Patient_ID   Age  Gender        Department Blood_Pressure  Heart_Rate  \
0     P07402  58.0  Female       Orthopedics         119/73        64.0   
1     P05835   NaN  Female  General Medicine         107/91        62.0   
2     P02123  47.0    Male       Orthopedics         148/98        83.0   
3     P08789   NaN    Male          Oncology         115/84        89.0   
4     P00305  47.0  Female       Orthopedics         128/62        77.0   
5     P02532  31.0  Female          Oncology         104/85        99.0   
6     P02996  49.0    MALE         Neurology         118/83       108.0   
7     P07660  32.0  Female        Pediatrics         122/89        64.0   
8     P08225  64.0    Male         Neurology        161/100        61.0   
9     P04449  56.0    Male  General Medicine         102/76        83.0   

   Temperature  Cholesterol  Glucose      Diagnosis Admission_Type  \
0        100.0        197.0     60.0  Heart Disease       Referral   
1   

In [16]:
# 7. Identify all columns containing missing values.
missing = df.isnull().sum()
print(missing[missing > 0])

# 8. Calculate the percentage of missing values in each column.
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 0].round(2))

# 9. Handle missing values in the numerical columns appropriately.

numerical_cols = ['Age', 'Heart_Rate', 'Temperature', 'Cholesterol', 'Glucose']

for col in numerical_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print(df[numerical_cols].isnull().sum())

# 10. Identify and remove duplicate records.
print('Rows before:', len(df))

df = df.drop_duplicates()

print('Rows after:', len(df))
# 11. Check the Age column for invalid values.
print('Age min:', df['Age'].min())
print('Age max:', df['Age'].max())

# Flag invalid ages using a reasonable human age range
invalid_age = df[(df['Age'] < 0) | (df['Age'] > 120)]
print('Number of invalid Age records:', len(invalid_age))
print(invalid_age['Age'].value_counts().sort_index())
# 12. Correct or handle patients whose age is outside a reasonable range.\
valid_median_age = df.loc[(df['Age'] >= 0) & (df['Age'] <= 120), 'Age'].median()

df.loc[(df['Age'] < 0) | (df['Age'] > 120), 'Age'] = valid_median_age

print('Invalid records remaining:', len(df[(df['Age'] < 0) | (df['Age'] > 120)]))
print('New Age range:', df['Age'].min(), '-', df['Age'].max())
# 13. Identify inconsistent values in categorical columns such as Gender and Department.
categorical_cols = ['Gender', 'Department', 'Diagnosis', 'Admission_Type', 'Readmission']

for col in categorical_cols:
    print(f'=== {col} ===')
    print(df[col].value_counts())
    print()
# 14. Standardize these categorical values. 

df['Gender'] = df['Gender'].str.strip().str.title()
df['Department'] = df['Department'].str.strip().str.title()

print(df['Gender'].value_counts())
print(df['Department'].value_counts())

Series([], dtype: int64)
Series([], dtype: float64)
Age            0
Heart_Rate     0
Temperature    0
Cholesterol    0
Glucose        0
dtype: int64
Rows before: 10000
Rows after: 10000
Age min: 1.0
Age max: 90.0
Number of invalid Age records: 0
Series([], Name: count, dtype: int64)
Invalid records remaining: 0
New Age range: 1.0 - 90.0
=== Gender ===
Gender
Male        4920
Female      4780
MALE         100
male         100
 FEMALE      100
Name: count, dtype: int64

=== Department ===
Department
Oncology            1662
Orthopedics         1644
Pediatrics          1614
Neurology           1612
General Medicine    1607
Cardiology          1561
 CARDIOLOGY          100
cardiology           100
neurology            100
Name: count, dtype: int64

=== Diagnosis ===
Diagnosis
Cancer           1279
Diabetes         1275
Asthma           1274
Infection        1272
Hypertension     1249
Migraine         1243
Fracture         1208
Heart Disease    1200
Name: count, dtype: int64

=== Admission

In [19]:
# 15. Create a new column called Age_Group:
# < 18 Child
# 18–40 Young Adult
# 41–60 Middle Age
# > 60 Senior
def age_group(age):
    if age < 18:
        return 'Child'
    elif age <= 40:
        return 'Young Adult'
    elif age <= 60:
        return 'Middle Age'
    else:
        return 'Senior'

df['Age_Group'] = df['Age'].apply(age_group)
print(df['Age_Group'].value_counts())

# 16. Separate Blood_Pressure into two columns:
# • Systolic
# • Diastolic
bp_split = df['Blood_Pressure'].str.split('/', expand=True)
df['Systolic'] = bp_split[0].astype(int)
df['Diastolic'] = bp_split[1].astype(int)

print(df[['Blood_Pressure', 'Systolic', 'Diastolic']].head())

# 17. Create a High_Risk column based on suitable clinical conditions.
high_risk_condition = (
    (df['Systolic'] >= 140) | (df['Diastolic'] >= 90) |
    (df['Cholesterol'] >= 240) |
    (df['Glucose'] >= 126) |
    (df['Heart_Rate'] > 100) | (df['Heart_Rate'] < 60) |
    (df['Temperature'] >= 100.4)
)

df['High_Risk'] = high_risk_condition.map({True: 'Yes', False: 'No'})
print(df['High_Risk'].value_counts())
# 18. Create a Cost_Per_Day column using treatment cost and length of stay. 
df['Cost_Per_Day'] = df['Treatment_Cost'] / df['Length_of_Stay']
print(df[['Treatment_Cost', 'Length_of_Stay', 'Cost_Per_Day']].head())

Age_Group
Middle Age     4377
Young Adult    2901
Senior         2243
Child           479
Name: count, dtype: int64
  Blood_Pressure  Systolic  Diastolic
0         119/73       119         73
1         107/91       107         91
2         148/98       148         98
3         115/84       115         84
4         128/62       128         62
High_Risk
Yes    6352
No     3648
Name: count, dtype: int64
   Treatment_Cost  Length_of_Stay  Cost_Per_Day
0        91183.33               5  18236.666000
1        71104.32               5  14220.864000
2       135197.73               7  19313.961429
3       146462.06               6  24410.343333
4        57510.66               6   9585.110000


In [36]:
# 19. Calculate the average age of patients.
print('Age Average:', df['Age'].mean())

# 20. Find the number of patients in each department.
print('Patient in each de[artment: ')
print(df['Department'].value_counts())
# 21. Calculate the average treatment cost for each department.
avg_cost = df.groupby('Department')['Treatment_Cost'].mean().sort_values(ascending=False)
print(avg_cost.round(2))

# 22. Find the average length of stay for each diagnosis.  
avg_cost = df.groupby('Diagnosis')['Length_of_Stay'].mean().sort_values(ascending=False)
print(avg_cost.round(2))
# 23. Identify the top 10 patients with the highest treatment cost.
top10 = df.sort_values('Treatment_Cost', ascending=False).head(10)[
    ['Patient_ID','Age','Department','Diagnosis','Treatment_Cost']
]
print(top10)

# 24. Find the department with the highest average treatment cost.
avg_cost = df.groupby('Department')['Treatment_Cost'].mean()
highest_dept = avg_cost.idxmax()
highest_amount = avg_cost.max()

print(f'Department with highest average treatment cost: {highest_dept} (₹{highest_amount:.2f})')

# 25. Calculate the percentage of patients who were readmitted.
readmit_patients = (df['Readmission'] == 'Yes').mean() * 100
print(f'Percentage readmitted: {readmit_patients:.2f}%')
# 26. Compare readmission rates across departments.
readmit_by_dept = (
    df.groupby('Department')['Readmission']
      .apply(lambda x: (x == 'Yes').mean() * 100)
      .sort_values(ascending=False)
)
print(readmit_by_dept.round(2))
# 27. Find the average cholesterol and glucose level by diagnosis.

avg_by_diagnosis = df.groupby('Diagnosis')[['Cholesterol', 'Glucose']].mean()
print(avg_by_diagnosis.round(2))
# 28. Identify patients with:
# • high glucose
# • high cholesterol
# • high heart rate

high_glucose = df[df['Glucose'] >= 126]
high_cholesterol = df[df['Cholesterol'] >= 240]
high_heart_rate = df[df['Heart_Rate'] > 100]

print('High Glucose patients:', len(high_glucose))
print('High Cholesterol patients:', len(high_cholesterol))
print('High Heart Rate patients:', len(high_heart_rate))

Age Average: 47.3914
Patient in each de[artment: 
Department
Cardiology          1761
Neurology           1712
Oncology            1662
Orthopedics         1644
Pediatrics          1614
General Medicine    1607
Name: count, dtype: int64
Department
Cardiology          102040.80
Neurology           101224.79
General Medicine    100711.12
Orthopedics         100302.54
Oncology            100039.38
Pediatrics           99036.60
Name: Treatment_Cost, dtype: float64
Diagnosis
Hypertension     5.10
Diabetes         5.08
Migraine         5.05
Fracture         5.03
Infection        5.02
Heart Disease    5.00
Cancer           5.00
Asthma           4.99
Name: Length_of_Stay, dtype: float64
      Patient_ID   Age   Department      Diagnosis  Treatment_Cost
1796      P09106  41.0    Neurology      Infection       315212.62
10066     P01900  42.0  Orthopedics      Infection       288519.47
4832      P06995  36.0     Oncology         Asthma       286702.48
7376      P09616  55.0    Neurology         